# Silver layer
Clean and model the bronze records into two tables: `silver_works` (one row per unique work)
and `silver_work_universities` (one row per work and university).

In [0]:
from pyspark.sql import functions as F

# Read the bronze table registered in Unity Catalog
df_bronze = spark.table("workspace.openalex.bronze_works")

In [0]:
%sql
SELECT
  COUNT(*)                      AS n_records,
  COUNT(DISTINCT id)            AS n_works,
  COUNT(*) - COUNT(DISTINCT id) AS n_extra_records
FROM workspace.openalex.bronze_works

In [0]:
%sql
-- For each work: how many rows it has and in how many university files it appears
WITH per_work AS (
  SELECT
    id,
    COUNT(*)                     AS n_rows,
    COUNT(DISTINCT _source_file) AS n_files
  FROM workspace.openalex.bronze_works
  GROUP BY id
)
SELECT n_rows, n_files, COUNT(*) AS n_works
FROM per_work
GROUP BY n_rows, n_files
ORDER BY n_rows, n_files

## silver_work_universities
One row per work and university. Built before deduplicating works, because the university is only known from the bronze source file.

In [0]:
# One row per (work, university) pair
df_work_universities = (
    df_bronze
    .select(
        F.regexp_replace("id", r"^https://openalex\.org/", "").alias("work_id"),
        F.regexp_replace("_source_file", r"\.jsonl$", "").alias("university"),
    )
    # Safety net: a pair must appear only once
    .dropDuplicates(["work_id", "university"])
)

display(df_work_universities.limit(5))

In [0]:
# Expected: 70,959 pairs and 66,972 distinct works
n_pairs = df_work_universities.count()
n_works = df_work_universities.select("work_id").distinct().count()
print(f"Pairs: {n_pairs:,}   Distinct works: {n_works:,}")

# Expected: UDC 15,544 · USC 33,709 · UVIGO 21,706
display(df_work_universities.groupBy("university").count().orderBy("university"))

In [0]:
# Every work_id must look like W123... and every university must be a known one.
# Nulls are checked explicitly: a comparison with null is neither true nor false.
invalid = df_work_universities.filter(
    F.col("work_id").isNull()
    | ~F.col("work_id").rlike(r"^W\d+$")
    | F.col("university").isNull()
    | ~F.col("university").isin("UDC", "USC", "UVIGO")
)
print(f"Invalid rows: {invalid.count()}")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.openalex.silver_work_universities (
  work_id    STRING NOT NULL COMMENT 'OpenAlex work ID, e.g. W2798336535',
  university STRING NOT NULL COMMENT 'UDC, USC or UVIGO',
  CONSTRAINT pk_work_universities PRIMARY KEY (work_id, university)
)
COMMENT 'One row per work and university (full counting): a work shared by several universities appears once for each.'

In [0]:
# Register the DataFrame so SQL can use it as the MERGE source
df_work_universities.createOrReplaceTempView("source_work_universities")

In [0]:
%sql
-- Upsert: insert new pairs, delete pairs that are no longer in the source
MERGE INTO workspace.openalex.silver_work_universities AS t
USING source_work_universities AS s
  ON t.work_id = s.work_id AND t.university = s.university
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [0]:
%sql
SELECT
  version,
  operation,
  operationMetrics['numTargetRowsInserted'] AS rows_inserted,
  operationMetrics['numTargetRowsDeleted']  AS rows_deleted
FROM (DESCRIBE HISTORY workspace.openalex.silver_work_universities)
ORDER BY version DESC
LIMIT 3

## silver_works
One row per unique work. Nested fields are flattened into typed columns following the data dictionary of the original project.

In [0]:
OPENALEX_PREFIX = r"^https://openalex\.org/"

# ISO 639-3 codes that have an ISO 639-1 equivalent (only "eng" appears in the data)
LANGUAGE_FIXES = {"eng": "en"}

# Flatten the nested record into one typed column per field
df_works_flat = (
    df_bronze
    .select(
        # Identifiers (URL prefixes removed; DOIs are case-insensitive, so lower-cased)
        F.regexp_replace("id", OPENALEX_PREFIX, "").alias("work_id"),
        F.lower(F.regexp_replace("doi", r"^https://doi\.org/", "")).alias("doi"),

        # Basic metadata
        F.col("title"),
        F.col("publication_year").cast("int").alias("year"),
        F.lower(F.trim("type")).alias("type"),
        F.lower(F.trim("language")).alias("language"),

        # Open access
        F.col("open_access.is_oa").alias("is_oa"),
        F.lower(F.trim("open_access.oa_status")).alias("oa_status"),

        # Source (journal or conference)
        F.regexp_replace("primary_location.source.id", OPENALEX_PREFIX, "").alias("source_id"),
        F.col("primary_location.source.display_name").alias("source_name"),
        F.col("primary_location.source.type").alias("source_type"),
        F.col("primary_location.source.host_organization_name").alias("publisher"),
        F.col("primary_location.source.issn_l").alias("issn_l"),
        # All ISSNs joined with ";" (null instead of an empty string when there are none)
        F.when(
            F.size("primary_location.source.issn") > 0,
            F.concat_ws(";", "primary_location.source.issn"),
        ).alias("issns"),

        # Subject classification
        F.col("primary_topic.domain.display_name").alias("domain"),
        F.col("primary_topic.field.display_name").alias("field"),
        F.col("primary_topic.subfield.display_name").alias("subfield"),

        # Citation impact
        F.col("cited_by_count").cast("int").alias("cited_by_count"),
        F.col("fwci"),
        F.col("citation_normalized_percentile.is_in_top_10_percent").alias("is_top10"),
        F.col("citation_normalized_percentile.is_in_top_1_percent").alias("is_top1"),

        # Collaboration
        F.col("countries_distinct_count").cast("int").alias("n_countries"),
        F.col("institutions_distinct_count").cast("int").alias("n_institutions"),
    )
    # International = more than one country (stays null when the count is missing)
    .withColumn("is_international", F.col("n_countries") > 1)
    
    # Normalise 3-letter language codes to their 2-letter form (eng -> en)
    .replace(LANGUAGE_FIXES, subset=["language"])
)

In [0]:
df_works_flat.printSchema()
display(df_works_flat.limit(5))

In [0]:
# Copies of a shared work must be identical; otherwise stop and decide which copy to keep
n_works = df_works_flat.select("work_id").distinct().count()
n_distinct_rows = df_works_flat.distinct().count()
print(f"Distinct work_id:   {n_works:,}")
print(f"Distinct full rows: {n_distinct_rows:,}")

assert n_distinct_rows == n_works, "Copies of a shared work differ: define a rule to choose one"

In [0]:
# Copies are identical (asserted above), so keeping any one of them loses nothing
df_works = df_works_flat.dropDuplicates(["work_id"])

### Data quality rules
Blocking rules send a work to `silver_works_rejected`; warning rules keep the work and record the issue in `dq_issues`.

In [0]:
VALID_TYPES = ["article", "review", "conference-paper"]
VALID_OA_STATUS = ["gold", "diamond", "hybrid", "bronze", "green", "closed"]
YEAR_FROM, YEAR_TO = 2015, 2025

# Blocking rules: the work cannot be used. A null value fails the rule.
BLOCKING_RULES = {
    "invalid_work_id":   F.col("work_id").isNull() | ~F.col("work_id").rlike(r"^W\d+$"),
    "year_out_of_range": F.col("year").isNull() | ~F.col("year").between(YEAR_FROM, YEAR_TO),
    "invalid_type":      F.col("type").isNull() | ~F.col("type").isin(VALID_TYPES),
}

# Warning rules: the work is kept and flagged. A null value does not raise a warning.
WARNING_RULES = {
    # Validity
    "invalid_doi":        ~F.col("doi").rlike(r"^10\.\d{4,9}/\S+$"),
    "invalid_language":   ~F.col("language").rlike(r"^[a-z]{2,3}$"),
    "unknown_oa_status":  ~F.col("oa_status").isin(VALID_OA_STATUS),
    "negative_citations": F.col("cited_by_count") < 0,
    "negative_fwci":      F.col("fwci") < 0,
    "title_has_markup":   F.col("title").rlike(r"&[a-zA-Z]+;|&#\d+;|</?[a-zA-Z][^>]*>|\$\$|\\[a-zA-Z]+"),
    # Consistency
    "oa_flag_mismatch":   F.col("is_oa") != (F.col("oa_status") != "closed"),
    "top1_not_in_top10":  F.col("is_top1") & ~F.col("is_top10"),
    "no_countries":       F.col("n_countries") < 1,
    "fewer_institutions_than_countries": F.col("n_institutions") < F.col("n_countries"),
    
    # Uniqueness (_works_with_doi is computed with a window before applying the rules)
    "duplicate_doi":      F.col("doi").isNotNull() & (F.col("_works_with_doi") > 1),
}

In [0]:
from pyspark.sql import Window

def rule_flags(rules):
    """Return an array with the names of the rules a row breaks (empty if none)."""
    return F.array_compact(
        F.array(*[F.when(condition, F.lit(name)) for name, condition in rules.items()])
    )


df_works_checked = (
    df_works
    # Number of works with the same DOI (window: counts within each DOI, keeps every row)
    .withColumn("_works_with_doi", F.count("*").over(Window.partitionBy("doi")))
    .withColumn("rejection_reasons", rule_flags(BLOCKING_RULES))
    .withColumn("dq_issues", rule_flags(WARNING_RULES))
    .drop("_works_with_doi")
)


df_works_valid = (
    df_works_checked
    .filter(F.size("rejection_reasons") == 0)
    .drop("rejection_reasons")
)

df_works_rejected = df_works_checked.filter(F.size("rejection_reasons") > 0)

print(f"Valid works:    {df_works_valid.count():,}")
print(f"Rejected works: {df_works_rejected.count():,}")

In [0]:
# Works per rule broken (a work can break several rules)
def count_by_rule(df, column):
    return (
        df.select(F.explode(column).alias("rule"))
        .groupBy("rule").count()
        .orderBy(F.desc("count"))
    )

display(count_by_rule(df_works_rejected, "rejection_reasons"))
display(count_by_rule(df_works_valid, "dq_issues"))

### Write the silver tables
`silver_works` is loaded with MERGE (only changed works are updated, detected with a row hash).
`silver_works_rejected` holds the current rejections and is overwritten on every run.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.openalex.silver_works (
  work_id          STRING  NOT NULL COMMENT 'OpenAlex work ID, e.g. W2798336535',
  doi              STRING  COMMENT 'DOI without the https://doi.org/ prefix, lower case',
  title            STRING,
  year             INT     NOT NULL COMMENT 'Publication year (2015-2025)',
  type             STRING  NOT NULL COMMENT 'article, review or conference-paper',
  language         STRING  COMMENT 'Detected by OpenAlex; unreliable for titles with scientific notation',
  is_oa            BOOLEAN,
  oa_status        STRING  COMMENT 'gold, diamond, hybrid, bronze, green or closed',
  source_id        STRING,
  source_name      STRING,
  source_type      STRING,
  publisher        STRING,
  issn_l           STRING  COMMENT 'Linking ISSN of the source',
  issns            STRING  COMMENT 'All ISSNs of the source, separated by ;',
  domain           STRING,
  field            STRING,
  subfield         STRING,
  cited_by_count   INT,
  fwci             DOUBLE  COMMENT 'Field-Weighted Citation Impact (1 = world average)',
  is_top10         BOOLEAN,
  is_top1          BOOLEAN,
  n_countries      INT,
  n_institutions   INT,
  is_international BOOLEAN COMMENT 'More than one country among the affiliations',
  dq_issues        ARRAY<STRING> NOT NULL COMMENT 'Names of the warning rules the work breaks; empty if none',
  _row_hash  STRING NOT NULL COMMENT 'SHA-256 of all other columns (data and quality flags), used to detect changes',
  CONSTRAINT pk_works PRIMARY KEY (work_id)
)
COMMENT 'One row per unique OpenAlex work (articles, reviews, conference papers) of UDC, USC or UVIGO, 2015-2025.'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.openalex.silver_works_rejected (
  work_id            STRING,
  doi                STRING,
  title              STRING,
  year               INT,
  type               STRING,
  language           STRING,
  is_oa              BOOLEAN,
  oa_status          STRING,
  source_id          STRING,
  source_name        STRING,
  source_type        STRING,
  publisher          STRING,
  issn_l             STRING,
  issns              STRING,
  domain             STRING,
  field              STRING,
  subfield           STRING,
  cited_by_count     INT,
  fwci               DOUBLE,
  is_top10           BOOLEAN,
  is_top1            BOOLEAN,
  n_countries        INT,
  n_institutions     INT,
  is_international   BOOLEAN,
  dq_issues          ARRAY<STRING>,
  rejection_reasons ARRAY<STRING> NOT NULL COMMENT 'Names of the blocking rules the work breaks'
)
COMMENT 'Works that fail a blocking data quality rule, with the reasons. Overwritten on every run.'

In [0]:
# Columns that define the content of a row: everything except the hash itself
HASH_COLUMNS = df_works_valid.columns

# Row hash: any change in any column (data or quality flags) changes the hash
df_works_source = df_works_valid.withColumn(
    "_row_hash",
    F.sha2(F.to_json(F.struct(*HASH_COLUMNS)), 256),
)

df_works_source.createOrReplaceTempView("source_works")

# Rejections are a snapshot of the current run: overwrite the table (its schema is kept)
df_works_rejected.write.mode("overwrite").saveAsTable("workspace.openalex.silver_works_rejected")

In [0]:
%sql
-- Upsert: update changed works, insert new ones, delete works no longer in the source
MERGE INTO workspace.openalex.silver_works AS t
USING source_works AS s
  ON t.work_id = s.work_id
WHEN MATCHED AND t._row_hash <> s._row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [0]:
%sql
SELECT
  version,
  operation,
  operationMetrics['numTargetRowsInserted'] AS rows_inserted,
  operationMetrics['numTargetRowsUpdated']  AS rows_updated,
  operationMetrics['numTargetRowsDeleted']  AS rows_deleted
FROM (DESCRIBE HISTORY workspace.openalex.silver_works)
ORDER BY version DESC
LIMIT 3

### Quality summary
Completeness (share of missing values per column) and uniqueness of DOIs in the saved `silver_works` table.

In [0]:
df_silver_works = spark.table("workspace.openalex.silver_works")

# Business columns only (dq_issues and _row_hash are never null)
columns = [c for c in df_silver_works.columns if c not in ("dq_issues", "_row_hash")]

# One pass over the table: % of missing values in every column
row = df_silver_works.select(
    [F.round(100 * (1 - F.count(c) / F.count("*")), 2).alias(c) for c in columns]
).first()

completeness = spark.createDataFrame(
    [(c, float(row[c])) for c in columns], ["column", "pct_missing"]
)
display(completeness.orderBy(F.desc("pct_missing")))

In [0]:
%sql
-- Uniqueness: the same DOI under several OpenAlex work IDs
SELECT
  doi,
  COUNT(*)              AS n_works,
  collect_list(work_id) AS work_ids
FROM workspace.openalex.silver_works
WHERE doi IS NOT NULL
GROUP BY doi
HAVING COUNT(*) > 1
ORDER BY n_works DESC, doi